In [32]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('../..')))

In [33]:
from src.patient_data_dispatcher import PatientDataDispatcher, PatientDataType
from src.core.enums import MileStone
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

In [34]:
dmo_features = [
    "wb_all_sum_d",
    "walkdur_all_sum_d",
    "wbsteps_all_sum_d",
    "wbdur_all_avg_d",
    "wbdur_all_p90_d",
    "wbdur_all_var_d",
    "cadence_all_avg_d",
    "strdur_all_avg_d",
    "cadence_all_var_d",
    "strdur_all_var_d",
    "ws_1030_avg_d",
    "strlen_1030_avg_d",
    "wb_10_sum_d",
    "ws_10_p90_d",
    "wb_30_sum_d",
    "ws_30_avg_d",
    "strlen_30_avg_d",
    "cadence_30_avg_d",
    "strdur_30_avg_d",
    "ws_30_p90_d",
    "cadence_30_p90_d",
    "ws_30_var_d",
    "strlen_30_var_d",
    "wb_60_sum_d",
    "total_worn_h_d",
    "total_worn_during_waking_h_d",
]

In [35]:
pdd = PatientDataDispatcher("../../config/config.yaml", dmo_features, MileStone.ALL)
ids = list(set(pdd.metadata["Local.Participant"].to_list()))
dmo_data, dmo_labels = pdd.get_patient_data(PatientDataType.MILESTONE, ids=ids)

In [36]:
# remove patients that don't have a full dataset
patient_indexs = []
patient, visit, day, pat_features = dmo_data.shape
for p in range(patient):
    all_visits = True
    for v in range(visit):
        data = dmo_data[p, v]
        label = dmo_labels[p, v]
        if (data == -1.0).any() or label == -1.0:
            all_visits = False

    if all_visits:
        patient_indexs.append(p)

dmo_data = dmo_data[patient_indexs]
dmo_labels = dmo_labels[patient_indexs]

In [37]:
from sklearn.feature_selection import mutual_info_classif

In [38]:
n_patients, n_visit, n_label = dmo_labels.shape
n_patients, n_visit, n_day, n_features = dmo_data.shape

# stack all patient data into 2d matrix
pat_features = dmo_data.reshape(n_patients * n_visit * n_day, n_features)
train_labels = dmo_labels.reshape(n_patients * n_visit, n_label)

In [40]:
from sklearn.preprocessing import MinMaxScaler

In [42]:
scaler = MinMaxScaler()
pat_features= scaler.fit_transform(pat_features)

In [39]:
pat_features = pd.DataFrame(pat_features, columns=dmo_features)

# Dmo features sorted by variance

In [43]:
pat_features.var().sort_values(ascending=False)

AttributeError: 'numpy.float64' object has no attribute 'sort_values'